# 11 — Self-reportable engineered feature comparison

This comparison uses only the project canonical 16 self-reportable features plus four Python-engineered composites. It deliberately selects only the source columns mapped by `train_pcos_model.py`; it does **not** reuse the reference notebook’s broad feature frame (which includes blood, vital-sign, and sensitive fields).

All candidates share one stratified 80/20 holdout (`random_state=42`). CV is performed only on the outer training split. Candidate B's decision threshold is tuned on an internal validation split of the outer training data and the holdout is evaluated once. Composite inputs are deterministic transforms of approved self-reportable inputs; they do not add a new measurement.

In [1]:
from pathlib import Path
import json, warnings
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline

SEED = 42
HERE = Path.cwd().resolve()
ML_DIR = next((p for p in [HERE, *HERE.parents] if (p / 'pcos_model_features.json').exists()), None)
if ML_DIR is None:
    raise FileNotFoundError('Run this notebook from its ml/notebooks directory or an ancestor of ml/.')
DATA_PATH = ML_DIR / 'data' / 'PCOS_data_without_infertility.xlsx'
MODEL_PATH = ML_DIR / 'pcos_rf_model.pkl'
FEATURES_PATH = ML_DIR / 'pcos_model_features.json'
BASE_FEATURES = json.loads(FEATURES_PATH.read_text(encoding='utf-8'))
ENGINEERED = ['androgen_score', 'metabolic_score', 'physical_risk', 'cycle_severity']
EXPANDED_FEATURES = BASE_FEATURES + ENGINEERED
print('ML directory:', ML_DIR)
print('Workbook:', DATA_PATH)
print('Canonical features:', len(BASE_FEATURES), BASE_FEATURES)
print('Expanded features:', len(EXPANDED_FEATURES))

ML directory: /home/spideyhack/Desktop/Projects/Pcos_pre_diagnosis/pcos-agent/backend/ml
Workbook: /home/spideyhack/Desktop/Projects/Pcos_pre_diagnosis/pcos-agent/backend/ml/data/PCOS_data_without_infertility.xlsx
Canonical features: 16 ['age', 'weight_kg', 'height_cm', 'cycle_length_days', 'weight_gain', 'hair_growth', 'skin_darkening', 'hair_loss', 'acne', 'fast_food', 'regular_exercise', 'hip_inch', 'waist_inch', 'cycle_irregular', 'bmi', 'waist_hip_ratio']
Expanded features: 20


In [2]:
# Match the source-column mapping and base derivations in train_pcos_model.py.
# Select only these allowed raw source columns before any modeling.
KEEP_RAW = {
    'PCOS (Y/N)': 'pcos', 'Age (yrs)': 'age', 'Weight (Kg)': 'weight_kg',
    'Height(Cm)': 'height_cm', 'Cycle(R/I)': 'cycle_code',
    'Cycle length(days)': 'cycle_length_days', 'Weight gain(Y/N)': 'weight_gain',
    'hair growth(Y/N)': 'hair_growth', 'Skin darkening (Y/N)': 'skin_darkening',
    'Hair loss(Y/N)': 'hair_loss', 'Pimples(Y/N)': 'acne', 'Fast food (Y/N)': 'fast_food',
    'Reg.Exercise(Y/N)': 'regular_exercise', 'Hip(inch)': 'hip_inch', 'Waist(inch)': 'waist_inch',
}
raw = pd.read_excel(DATA_PATH, sheet_name='Full_new')
raw.columns = [str(c).strip() for c in raw.columns]
missing_columns = sorted(set(KEEP_RAW) - set(raw.columns))
if missing_columns:
    raise ValueError(f'Required canonical input columns missing: {missing_columns}')
data = raw[list(KEEP_RAW)].rename(columns=KEEP_RAW).copy()
data['cycle_irregular'] = (data['cycle_code'] != 2).astype(int)
data = data.drop(columns='cycle_code')
data['bmi'] = data['weight_kg'] / (data['height_cm'] / 100) ** 2
data['waist_hip_ratio'] = data['waist_inch'] / data['hip_inch']
data['pcos'] = data['pcos'].astype(int)
# Preserve exact canonical JSON order, independent of dataframe insertion order.
base = data[BASE_FEATURES].copy()
y = data['pcos'].copy()
print('Rows:', len(data), '| class counts:', y.value_counts().sort_index().to_dict())
print('Missing base values:', base.isna().sum()[base.isna().sum() > 0].to_dict())
print('Selected raw columns:', list(KEEP_RAW))
assert len(BASE_FEATURES) == 16 and len(EXPANDED_FEATURES) == 20

Rows: 541 | class counts: {0: 364, 1: 177}
Missing base values: {'fast_food': 1}
Selected raw columns: ['PCOS (Y/N)', 'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'Cycle(R/I)', 'Cycle length(days)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)', 'Hip(inch)', 'Waist(inch)']


In [3]:
def add_composites(frame):
    out = frame.copy()
    out['androgen_score'] = out['hair_growth'] + out['skin_darkening'] + out['acne'] + out['hair_loss']
    out['metabolic_score'] = out['weight_gain'] + out['fast_food'] + (out['bmi'] > 25).astype(int)
    out['physical_risk'] = out['bmi'] * out['waist_hip_ratio']
    out['cycle_severity'] = out['cycle_irregular'] * out['cycle_length_days']
    return out[EXPANDED_FEATURES]

X_train_idx, X_test_idx, y_train, y_test = train_test_split(
    np.arange(len(y)), y, test_size=0.2, stratify=y, random_state=SEED
)
X_train_raw = base.iloc[X_train_idx].copy()
X_test_raw = base.iloc[X_test_idx].copy()
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# The canonical training script historically imputed before splitting. For exact evaluation
# of that saved artifact, reproduce its one missing binary fast_food value (mode) globally.
# New candidates use train-only fitted imputation to avoid validation/test leakage.
legacy_filled = base.copy()
for col in legacy_filled:
    if legacy_filled[col].isna().any():
        vals = legacy_filled[col].dropna()
        fill = vals.mode().iloc[0] if vals.isin([0, 1]).all() else vals.median()
        legacy_filled[col] = legacy_filled[col].fillna(fill)
print('Canonical legacy imputation (matching trainer):', {
    c: float(legacy_filled[c].iloc[base[c].isna()].iloc[0]) for c in base if base[c].isna().any()
})
print('Outer split:', len(X_train_idx), 'train /', len(X_test_idx), 'test')

Canonical legacy imputation (matching trainer): {'fast_food': 1.0}
Outer split: 432 train / 109 test


In [4]:
# Baseline artifact is the existing canonical 16-feature model.
canonical = joblib.load(MODEL_PATH)
canonical_features = list(getattr(canonical, 'feature_names_in_', []))
if canonical_features != BASE_FEATURES:
    raise ValueError('Canonical model feature_names_in_ differs from the JSON order')
print('Baseline model:', canonical.__class__.__name__)
print('Baseline parameters:', {k: canonical.get_params().get(k) for k in ['n_estimators','max_depth','min_samples_leaf','class_weight','random_state']})
print('Note: this artifact has 400 trees, while the reference training script config specifies 300; evaluation uses the loaded existing artifact as requested.')

candidate_a = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=4,
    class_weight='balanced', random_state=SEED, n_jobs=1,
)

def make_candidate_b():
    rf = RandomForestClassifier(n_estimators=300, max_depth=12, random_state=SEED, n_jobs=1)
    et = ExtraTreesClassifier(n_estimators=300, max_depth=12, random_state=SEED, n_jobs=1)
    voting = VotingClassifier(estimators=[('rf', rf), ('extra_trees', et)], voting='soft', n_jobs=1)
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('borderline_smote', BorderlineSMOTE(random_state=SEED)),
        ('voting', voting),
    ])

# Prepare new candidate features after split, fitting imputation exclusively on training data.
# The training split has only one missing input; median imputation is fit there and reused.
imputer_new = SimpleImputer(strategy='median')
X_train_base_imp = pd.DataFrame(imputer_new.fit_transform(X_train_raw), columns=BASE_FEATURES)
X_test_base_imp = pd.DataFrame(imputer_new.transform(X_test_raw), columns=BASE_FEATURES)
X_train_20 = add_composites(X_train_base_imp)
X_test_20 = add_composites(X_test_base_imp)
print('Candidate feature shapes:', X_train_20.shape, X_test_20.shape)

Baseline model: RandomForestClassifier
Baseline parameters: {'n_estimators': 400, 'max_depth': 6, 'min_samples_leaf': 4, 'class_weight': 'balanced', 'random_state': 42}
Note: this artifact has 400 trees, while the reference training script config specifies 300; evaluation uses the loaded existing artifact as requested.
Candidate feature shapes: (432, 20) (109, 20)


In [5]:
# Candidate B threshold selection uses a validation subset from the outer training data only.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_20, y_train, test_size=0.20, stratify=y_train, random_state=SEED
)
b_threshold_model = make_candidate_b()
b_threshold_model.fit(X_fit, y_fit)
b_val_probs = b_threshold_model.predict_proba(X_val)[:, list(b_threshold_model.classes_).index(1)]
threshold_rows = []
for threshold in np.arange(0.10, 0.901, 0.01):
    val_pred = (b_val_probs >= threshold).astype(int)
    threshold_rows.append((f1_score(y_val, val_pred, zero_division=0), float(threshold),
                           recall_score(y_val, val_pred, zero_division=0)))
# Maximize PCOS F1; on ties choose the higher threshold to avoid unnecessary positives.
best_f1, B_THRESHOLD, val_recall = max(threshold_rows, key=lambda row: (row[0], row[1]))
print(f'Candidate B validation threshold={B_THRESHOLD:.2f}, validation F1={best_f1:.3f}, recall={val_recall:.3f}')

# Refit each candidate on the full outer training split. SMOTE remains confined to train.
candidate_a.fit(X_train_20, y_train)
candidate_b = make_candidate_b()
candidate_b.fit(X_train_20, y_train)


Candidate B validation threshold=0.46, validation F1=0.842, recall=0.857


,steps,"[('imputer', ...), ('borderline_smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](20,)","['age','weight_kg','height_cm',...,'metabolic_score','physical_risk', 'cycle_severity']"
n_features_in_,int,20
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None


In [6]:
# Stratified 5-fold CV over outer training rows. Candidate B's CV metrics use the
# standard 0.50 cutoff for comparability; its final holdout metrics use the train-only
# validation-tuned cutoff above. Every fold's BorderlineSMOTE runs inside its pipeline.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {'recall': 'recall', 'roc_auc': 'roc_auc'}

def cv_summary(estimator, X, y):
    scores = cross_validate(estimator, X, y, cv=cv, scoring=scoring, n_jobs=1, error_score='raise')
    return (scores['test_recall'].mean(), scores['test_recall'].std(ddof=1),
            scores['test_roc_auc'].mean(), scores['test_roc_auc'].std(ddof=1))

# Clone the saved baseline for CV; test scores use the actual trained canonical artifact.
baseline_cv = cv_summary(canonical, legacy_filled.iloc[X_train_idx][BASE_FEATURES].reset_index(drop=True), y_train)
a_cv = cv_summary(RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=4,
    class_weight='balanced', random_state=SEED, n_jobs=1), X_train_20, y_train)
b_cv = cv_summary(make_candidate_b(), X_train_20, y_train)

def test_metrics(name, estimator, X, threshold=0.5, classes_model=None):
    positive_idx = list(estimator.classes_).index(1)
    probs = estimator.predict_proba(X)[:, positive_idx]
    pred = (probs >= threshold).astype(int)
    return {
        'Model': name,
        'CV recall mean (std)': f'{cvvals[name][0]:.3f} ({cvvals[name][1]:.3f})',
        'CV ROC-AUC mean (std)': f'{cvvals[name][2]:.3f} ({cvvals[name][3]:.3f})',
        'Test accuracy': accuracy_score(y_test, pred),
        'Test precision': precision_score(y_test, pred, zero_division=0),
        'Test recall': recall_score(y_test, pred, zero_division=0),
        'Test F1': f1_score(y_test, pred, zero_division=0),
        'Test ROC-AUC': roc_auc_score(y_test, probs),
        'Confusion matrix [ [TN,FP], [FN,TP] ]': confusion_matrix(y_test, pred, labels=[0,1]).tolist(),
    }

cvvals = {'Canonical 16-feature baseline': baseline_cv,
          'Candidate A: engineered Random Forest': a_cv,
          'Candidate B: SMOTE soft voting': b_cv}
# Canonical test uses legacy-compatible pre-split imputation exactly as the canonical trainer did.
X_test_baseline = legacy_filled.iloc[X_test_idx][BASE_FEATURES].reset_index(drop=True)
rows = [
    test_metrics('Canonical 16-feature baseline', canonical, X_test_baseline, 0.5),
    test_metrics('Candidate A: engineered Random Forest', candidate_a, X_test_20, 0.5),
    test_metrics('Candidate B: SMOTE soft voting', candidate_b, X_test_20, B_THRESHOLD),
]
comparison = pd.DataFrame(rows).set_index('Model')
# Numeric columns rounded for presentation; confusion matrices and CV strings remain readable.
for col in ['Test accuracy','Test precision','Test recall','Test F1','Test ROC-AUC']:
    comparison[col] = comparison[col].map(lambda value: round(float(value), 3))
display(comparison)
print('RECALL_FIRST_RANKING')
print(comparison[['CV recall mean (std)','Test recall','Test precision','Test ROC-AUC']].to_string())

,CV recall mean (std),CV ROC-AUC mean (std),Test accuracy,Test precision,Test recall,Test F1,Test ROC-AUC,"Confusion matrix [ [TN,FP], [FN,TP] ]"
Model,,,,,,,,
Canonical 16-feature baseline,0.801 (0.054),0.881 (0.013),0.853,0.794,0.75,0.771,0.884,"[[66, 7], [9, 27]]"
Candidate A: engineered Random Forest,0.780 (0.077),0.892 (0.025),0.862,0.818,0.75,0.783,0.881,"[[67, 6], [9, 27]]"
Candidate B: SMOTE soft voting,0.710 (0.101),0.891 (0.030),0.853,0.794,0.75,0.771,0.911,"[[66, 7], [9, 27]]"


RECALL_FIRST_RANKING
                                      CV recall mean (std)  Test recall  Test precision  Test ROC-AUC
Model                                                                                                
Canonical 16-feature baseline                0.801 (0.054)         0.75           0.794         0.884
Candidate A: engineered Random Forest        0.780 (0.077)         0.75           0.818         0.881
Candidate B: SMOTE soft voting               0.710 (0.101)         0.75           0.794         0.911


## Recommendation

**Keep the existing canonical 16-feature Random Forest as the deployed model for this comparison.** It has the strongest mean PCOS-positive 5-fold training recall (0.801 ± 0.054). All candidates tie on held-out recall (0.750), so neither engineered candidate demonstrates an observed holdout-recall gain. Candidate A improves held-out precision slightly (0.818 vs. 0.794) but has lower CV recall (0.780); Candidate B has the best held-out ROC-AUC (0.911) but substantially lower CV recall (0.710). For a screening tool, the recall-first evidence favors retaining the canonical model.

The four composites are still compliant because they use only the approved self-reportable values. They are useful session annotations and comparison features, but these results do not justify changing the production model’s feature contract. The backend may compute them in Python for session state while continuing to send exactly the canonical 16 columns to the deployed model. This is a small single-dataset comparison, not evidence of clinical performance; any model change should be validated on independent data.